<a href="https://colab.research.google.com/github/jacobdawson093-tech/Montgomery-County-and-Bias-Incidents-Analysis/blob/main/eda/notebooks/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ingestion

In [13]:
import requests
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import datetime
import os
os.makedirs("graphs", exist_ok=True)

url = "https://api.census.gov/data/2024/acs/acs5"
response = requests.get(url)
data = response.json()
url2 = "https://data.montgomerycountymd.gov/api/v3/views/7bhj-887p/query.json?app_token=8kUbrmGzgoxqe4z7C91iV3wmC"
response2 = requests.get(url2)
data2 = response2.json()

df1 = pd.DataFrame(data)

df2 = pd.DataFrame(data2)
df2.head()

,:id,:version,:created_at,:updated_at,id,incident_date,district,bias_code,bias,status,victim_type,no_of_suspects,suspects_less_than_18_years,unknown,no_of_victims,suspects_36_45_years_old,suspects_55_years_old,bias_code_2,suspects_18_35_years_old,suspects_46_55_years_old
0,row-jqk7~r9v8~gn9q,rv-7acy.s23a-yq4u,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015370,2026-04-10T00:00:00.000,5D,Anti-Black,Vandalism,Closed-Admin,School/College,1,1,Known,NaN,NaN,NaN,NaN,NaN,NaN
1,row-ac27~z3ht~tkpj,rv-sdjt_wdbv-ffdb,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015219,2026-04-09T00:00:00.000,3D,Anti-Jewish,Vandalism,Open,School/College,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN
2,row-rnxy~y5m6.vfdz,rv-ywnn~vf3n~jwvz,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015206,2026-04-09T00:00:00.000,2D,Anti-Jewish,Verbal Intimidation/Simple Assault,Open,Individual(s),1,NaN,Unknown,1,NaN,NaN,NaN,NaN,NaN
3,row-5vip~qvx8-h56u,rv-c22m-iy9x~rpi6,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015047,2026-04-08T00:00:00.000,4D,Anti-Black,Verbal Intimidation/Simple Assault,Closed-Admin,School/College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,row-kt32_377n~ey23,rv-3inw-3eby-w733,2026-04-10T07:11:08.884Z,2026-04-10T07:11:08.884Z,260013584,2026-03-31T00:00:00.000,6D,Anti-Jewish,Vandalism,Closed-Admin,School/College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Convert 'incident_date' to datetime
df2['incident_date'] = pd.to_datetime(df2['incident_date'])

# Create a binary 'resolved' column from 'status'
df2['resolved'] = df2['status'].apply(lambda x: 1 if x in ['Closed-Admin', 'Closed-Investigation', 'Closed-Exception', 'Closed-Arrest'] else 0)

# Convert suspect age columns to numeric, coercing errors to NaN
df2['suspects_less_than_18_years'] = pd.to_numeric(df2['suspects_less_than_18_years'], errors='coerce').fillna(0).astype(int)
df2['suspects_18_35_years_old'] = pd.to_numeric(df2['suspects_18_35_years_old'], errors='coerce').fillna(0).astype(int)
df2['suspects_36_45_years_old'] = pd.to_numeric(df2['suspects_36_45_years_old'], errors='coerce').fillna(0).astype(int)
df2['suspects_46_55_years_old'] = pd.to_numeric(df2['suspects_46_55_years_old'], errors='coerce').fillna(0).astype(int)
df2['suspects_55_years_old'] = pd.to_numeric(df2['suspects_55_years_old'], errors='coerce').fillna(0).astype(int)

# Create a binary variable for incidents with suspects under 18
df2['under_18_suspect'] = df2['suspects_less_than_18_years'].apply(lambda x: 1 if x > 0 else 0)

#Create bias category to seperate religious bias to racial/ethnic biases
def categorize_bias(bias_code):
    religious_biases = ['Anti-Jewish', 'Anti-Muslim', 'Anti-Catholic', 'Anti-Sikh', 'Anti-Hindhu', 'Anti-Protestant', 'Anti-Buddhist']
    racial_ethnic_biases = ['Anti-Black', 'Anti-White', 'Anti-Asian', 'Anti-Hispanic', 'Anti-Multi-Racial', 'Anti-American Indian', 'Anti-Other Race/Ethnicity']

    if bias_code in religious_biases:
        return 'Religious Bias'
    elif bias_code in racial_ethnic_biases:
        return 'Racial/Ethnic Bias'
    else:
        return 'Other Bias' # For categories not explicitly religious or racial/ethnic

df2['bias_category'] = df2['bias_code'].apply(categorize_bias)

In [15]:
print("\n--- Frequency Distributions for Categorical Variables ---")
categorical_cols = [
    'district', 'bias_code', 'bias_code_2', 'bias', 'status', 'victim_type'
]

for col in categorical_cols:
    print(f"\nFrequency distribution for '{col}':")
    print(df2[col].value_counts(dropna=False))
    print(f"Percentage distribution for '{col}':")
    print(df2[col].value_counts(normalize=True, dropna=False) * 100)


--- Frequency Distributions for Categorical Variables ---

Frequency distribution for 'district':
district
2D      566
4D      384
1D      303
5D      269
3D      245
6D      177
RCPD    116
GCPD    101
TPPD     26
Name: count, dtype: int64
Percentage distribution for 'district':
district
2D      25.880201
4D      17.558299
1D      13.854595
5D      12.299954
3D      11.202561
6D       8.093278
RCPD     5.304070
GCPD     4.618198
TPPD     1.188843
Name: proportion, dtype: float64

Frequency distribution for 'bias_code':
bias_code
Anti-Jewish                   768
Anti-Black                    697
Anti-Homosexual               154
Anti-Asian                    105
Anti-Hispanic                  98
Anti-Multi-Racial              66
Anti-Islamic                   60
Anti-White                     46
Anti-Male Homosexual           44
Anti-Transgender               40
Anti-Other Ethnicity           22
Anti-Arab                      11
Anti-Mental Disability         10
Anti-Catholic        

In [16]:
print("\n--- Time-based Analysis of Incident Dates ---")
# Convert 'incident_date' to datetime objects
df2['incident_date'] = pd.to_datetime(df2['incident_date'], errors='coerce')

# Drop rows where incident_date could not be parsed
df2.dropna(subset=['incident_date'], inplace=True)

# Extract year and month for time-based analysis
df2['incident_year'] = df2['incident_date'].dt.year
df2['incident_month'] = df2['incident_date'].dt.month

print("Incidents per Year:")
print(df2['incident_year'].value_counts().sort_index())

print("\nIncidents per Month (across all years):")
print(df2['incident_month'].value_counts().sort_index())


--- Time-based Analysis of Incident Dates ---
Incidents per Year:
incident_year
2016     98
2017    122
2018     93
2019    114
2020    117
2021    144
2022    160
2023    466
2024    485
2025    332
2026     56
Name: count, dtype: int64

Incidents per Month (across all years):
incident_month
1     174
2     229
3     235
4     185
5     212
6     183
7     107
8     128
9     173
10    194
11    201
12    166
Name: count, dtype: int64


## Consolidated Statistical Findings

### 1. Initial Data Overview (First 5 Rows)

In [17]:
display(df2.head())

,:id,:version,:created_at,:updated_at,id,incident_date,district,bias_code,bias,status,...,suspects_36_45_years_old,suspects_55_years_old,bias_code_2,suspects_18_35_years_old,suspects_46_55_years_old,resolved,under_18_suspect,bias_category,incident_year,incident_month
0,row-jqk7~r9v8~gn9q,rv-7acy.s23a-yq4u,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015370,2026-04-10,5D,Anti-Black,Vandalism,Closed-Admin,...,0,0,NaN,0,0,1,1,Racial/Ethnic Bias,2026,4
1,row-ac27~z3ht~tkpj,rv-sdjt_wdbv-ffdb,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015219,2026-04-09,3D,Anti-Jewish,Vandalism,Open,...,0,0,NaN,0,0,0,0,Religious Bias,2026,4
2,row-rnxy~y5m6.vfdz,rv-ywnn~vf3n~jwvz,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015206,2026-04-09,2D,Anti-Jewish,Verbal Intimidation/Simple Assault,Open,...,0,0,NaN,0,0,0,0,Religious Bias,2026,4
3,row-5vip~qvx8-h56u,rv-c22m-iy9x~rpi6,2026-05-01T07:11:17.513Z,2026-05-01T07:11:17.513Z,260015047,2026-04-08,4D,Anti-Black,Verbal Intimidation/Simple Assault,Closed-Admin,...,0,0,NaN,0,0,1,0,Racial/Ethnic Bias,2026,4
4,row-kt32_377n~ey23,rv-3inw-3eby-w733,2026-04-10T07:11:08.884Z,2026-04-10T07:11:08.884Z,260013584,2026-03-31,6D,Anti-Jewish,Vandalism,Closed-Admin,...,0,0,NaN,0,0,1,0,Religious Bias,2026,3


### 2. Summary Statistics for Numerical Variables

In [18]:
numerical_cols = [
    'no_of_victims', 'no_of_suspects', 'suspects_less_than_18_years',
    'suspects_18_35_years_old', 'suspects_36_45_years_old',
    'suspects_46_55_years_old', 'suspects_55_years_old'
]
display(df2[numerical_cols].describe())

,suspects_less_than_18_years,suspects_18_35_years_old,suspects_36_45_years_old,suspects_46_55_years_old,suspects_55_years_old
count,2187.000000,2187.000000,2187.000000,2187.000000,2187.000000
mean,0.332419,0.065844,0.040695,0.031093,0.039323
std,0.707861,0.272663,0.204454,0.178801,0.201343
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,0.000000,0.000000,0.000000
max,9.000000,3.000000,2.000000,2.000000,2.000000


### 3. Frequency and Percentage Distributions for Categorical Variables

In [19]:
categorical_cols = [
    'district', 'bias_code', 'bias_code_2', 'bias', 'status', 'victim_type'
]

for col in categorical_cols:
    print(f"\n--- '{col}' Distribution ---")
    # Combine value counts and normalized percentages into a single DataFrame for display
    freq_df = pd.DataFrame({
        'Count': df2[col].value_counts(dropna=False),
        'Percentage': df2[col].value_counts(normalize=True, dropna=False) * 100
    })
    display(freq_df.sort_values(by='Count', ascending=False))


--- 'district' Distribution ---


,Count,Percentage
district,,
2D,566,25.880201
4D,384,17.558299
1D,303,13.854595
5D,269,12.299954
3D,245,11.202561
6D,177,8.093278
RCPD,116,5.304070
GCPD,101,4.618198
TPPD,26,1.188843



--- 'bias_code' Distribution ---


,Count,Percentage
bias_code,,
Anti-Jewish,768,35.116598
Anti-Black,697,31.870142
Anti-Homosexual,154,7.041610
Anti-Asian,105,4.801097
Anti-Hispanic,98,4.481024
Anti-Multi-Racial,66,3.017833
Anti-Islamic,60,2.743484
Anti-White,46,2.103338
Anti-Male Homosexual,44,2.011888



--- 'bias_code_2' Distribution ---


...

,Count,Percentage
bias,,
Vandalism,809,36.991312
Verbal Intimidation/Simple Assault,627,28.669410
Written Intimidation/Simple Assault,269,12.299954
Other,173,7.910380
Assault (simple),129,5.898491
Social Media,44,2.011888
Flyer Left Behind,39,1.783265
Physical Intimidation/Simple Assault,33,1.508916
Assault (physical),25,1.143118



--- 'status' Distribution ---


,Count,Percentage
status,,
Open,1019,46.593507
Closed-Admin,675,30.864198
N/A,191,8.733425
Closed-Arrest,123,5.624143
Inactive,94,4.298125
Closed-Exception,43,1.966164
UNF,29,1.326017
RTOJ,8,0.365798
NaN,5,0.228624



--- 'victim_type' Distribution ---


,Count,Percentage
victim_type,,
Individual(s),1187,54.275263
School/College,556,25.422954
Society,164,7.498857
Religious Organization,129,5.898491
Business/Financial Institution,91,4.160951
Government,27,1.234568
Other,19,0.868770
NaN,14,0.640146


### 4. Time-based Analysis

#### Incidents per Year

In [20]:
incidents_per_year_df = df2['incident_year'].value_counts().sort_index().reset_index()
incidents_per_year_df.columns = ['Year', 'Incident Count']
display(incidents_per_year_df)

,Year,Incident Count
0,2016,98
1,2017,122
2,2018,93
3,2019,114
4,2020,117
5,2021,144
6,2022,160
7,2023,466
8,2024,485
9,2025,332


#### Incidents per Month (across all years)

In [21]:
incidents_per_month_df = df2['incident_month'].value_counts().sort_index().reset_index()
incidents_per_month_df.columns = ['Month', 'Incident Count']
display(incidents_per_month_df)

,Month,Incident Count
0,1,174
1,2,229
2,3,235
3,4,185
4,5,212
5,6,183
6,7,107
7,8,128
8,9,173
9,10,194


In [22]:
import os
output_dir = 'eda_statistical_findings'
os.makedirs(output_dir, exist_ok=True)

df2.head().to_csv(os.path.join(output_dir, 'initial_data_head.csv'), index=False)
df2[numerical_cols].describe().to_csv(os.path.join(output_dir, 'numerical_summary.csv'))

for col in categorical_cols:
    freq_df = pd.DataFrame({
        'Count': df2[col].value_counts(dropna=False),
        'Percentage': df2[col].value_counts(normalize=True, dropna=False) * 100
    })
    freq_df.sort_values(by='Count', ascending=False).to_csv(os.path.join(output_dir, f'{col}_distribution.csv'))

incidents_per_year_df.to_csv(os.path.join(output_dir, 'incidents_per_year.csv'), index=False)
incidents_per_month_df.to_csv(os.path.join(output_dir, 'incidents_per_month.csv'), index=False)

print(f"All statistical findings saved to the '{output_dir}' directory.")

All statistical findings saved to the 'eda_statistical_findings' directory.


In [23]:
import os

# Define the directory and output path for the markdown file
output_dir = 'eda_statistical_findings'
markdown_output_path = os.path.join(output_dir, 'all_eda_findings.md')

# Prepare the content for the Markdown file
markdown_content = "# Consolidated EDA Statistical Findings\n\n"

# 1. Initial Data Overview
markdown_content += "## 1. Initial Data Overview (First 5 Rows)\n\n"
markdown_content += df2.head().to_markdown(index=False)
markdown_content += "\n\n"

# 2. Summary Statistics for Numerical Variables
markdown_content += "## 2. Summary Statistics for Numerical Variables\n\n"
markdown_content += df2[numerical_cols].describe().to_markdown()
markdown_content += "\n\n"

# 3. Frequency and Percentage Distributions for Categorical Variables
markdown_content += "## 3. Frequency and Percentage Distributions for Categorical Variables\n\n"
for col in categorical_cols:
    markdown_content += f"### '{col}' Distribution\n\n"
    freq_df = pd.DataFrame({
        'Count': df2[col].value_counts(dropna=False),
        'Percentage': df2[col].value_counts(normalize=True, dropna=False) * 100
    })
    markdown_content += freq_df.sort_values(by='Count', ascending=False).to_markdown()
    markdown_content += "\n\n"

# 4. Time-based Analysis - Incidents per Year
markdown_content += "## 4. Time-based Analysis\n\n"
markdown_content += "### Incidents per Year\n\n"
markdown_content += incidents_per_year_df.to_markdown(index=False)
markdown_content += "\n\n"

# 5. Time-based Analysis - Incidents per Month
markdown_content += "### Incidents per Month (across all years)\n\n"
markdown_content += incidents_per_month_df.to_markdown(index=False)
markdown_content += "\n\n"

# Save the Markdown content to a file
with open(markdown_output_path, 'w') as f:
    f.write(markdown_content)